<a href="https://colab.research.google.com/github/huyd073003/AAI2026/blob/ad_optimization_agent/ad_optimization_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path("/content")
df = pd.read_csv(BASE / "mock_ad_data.csv")

MIN_SHARE = 0.20
MAX_DAILY_SHIFT = 0.20
TOTAL_BUDGET = 300.0
LOOKBACK = 3

channels = list(df['channel'].unique())

df['ctr'] = df['clicks'] / df['impressions']
df['cvr'] = np.where(df['clicks'] > 0, df['conversions'] / df['clicks'], 0)
df['conv_per_dollar'] = np.where(df['spend'] > 0, df['conversions'] / df['spend'], 0)

dates = sorted(df['date'].unique())

alloc = {ch: TOTAL_BUDGET / len(channels) for ch in channels}
agent_logs = []
agent_daily = []

for idx, current_date in enumerate(dates):
    day_rows = df[df['date'] == current_date].set_index('channel')
    expected = {ch: alloc[ch] * float(day_rows.loc[ch, 'conv_per_dollar']) for ch in channels}

    agent_daily.append({
        'date': current_date,
        **{f'{ch}_budget': round(alloc[ch], 2) for ch in channels},
        'total_expected_conversions': round(sum(expected.values()), 2)
    })

    if idx == len(dates) - 1:
        break

    hist_dates = dates[max(0, idx - LOOKBACK + 1):idx + 1]
    hist = df[df['date'].isin(hist_dates)]

    scores = {}
    for ch in channels:
        h = hist[hist['channel'] == ch]
        clicks_per_dollar = h['clicks'].sum() / max(h['spend'].sum(), 1)
        conv_per_dollar = h['conversions'].sum() / max(h['spend'].sum(), 1)
        scores[ch] = 0.75 * conv_per_dollar + 0.25 * clicks_per_dollar / 10

    score_sum = sum(scores.values()) or 1
    target = {ch: TOTAL_BUDGET * scores[ch] / score_sum for ch in channels}

    new_alloc = {}
    reasons = []

    for ch in channels:
        lower = alloc[ch] * (1 - MAX_DAILY_SHIFT)
        upper = alloc[ch] * (1 + MAX_DAILY_SHIFT)
        bounded = min(max(target[ch], lower), upper)
        bounded = max(bounded, TOTAL_BUDGET * MIN_SHARE)
        new_alloc[ch] = bounded
        reasons.append(f"{ch}: target={target[ch]:.2f}, applied={bounded:.2f}, score={scores[ch]:.4f}")

    total = sum(new_alloc.values())
    if abs(total - TOTAL_BUDGET) > 1e-6:
        adjustable = [ch for ch in channels if new_alloc[ch] > TOTAL_BUDGET * MIN_SHARE + 0.01]
        excess = total - TOTAL_BUDGET
        if adjustable:
            adj_total = sum(new_alloc[ch] - TOTAL_BUDGET * MIN_SHARE for ch in adjustable)
            for ch in adjustable:
                room = new_alloc[ch] - TOTAL_BUDGET * MIN_SHARE
                new_alloc[ch] -= excess * (room / adj_total)

    alloc = {ch: round(new_alloc[ch], 2) for ch in channels}

    agent_logs.append({
        'decision_for_date': dates[idx + 1],
        'rationale': '; '.join(reasons),
        **{f'{ch}_new_budget': alloc[ch] for ch in channels}
    })

agent_daily_df = pd.DataFrame(agent_daily)
agent_logs_df = pd.DataFrame(agent_logs)

agent_daily_df.to_csv(BASE / "agent_daily_allocations.csv", index=False)
agent_logs_df.to_csv(BASE / "decision_log.csv", index=False)

print("Done")
print(agent_logs_df.head())

Done
  decision_for_date                                          rationale  \
0        2025-03-02  Search: target=160.75, applied=120.00, score=0...   
1        2025-03-03  Search: target=166.25, applied=150.19, score=0...   
2        2025-03-04  Search: target=164.41, applied=164.41, score=0...   
3        2025-03-05  Search: target=165.80, applied=165.80, score=0...   
4        2025-03-06  Search: target=161.55, applied=161.55, score=0...   

   Search_new_budget  Social_new_budget  Display_new_budget  
0             125.16              93.12               81.72  
1             145.34              89.57               65.09  
2             152.71              87.29               60.00  
3             152.56              87.44               60.00  
4             150.86              89.14               60.00  
